# <span style="font-width:bold; font-size: 3rem; color:#1EB182;">**Garmin Companion**</span><span style="font-width:bold; font-size: 3rem; color:#333;"> - 01: Feature Pipeline</span>

<span style="font-width:bold; font-size: 1.4rem;">Ingest Garmin data and build the raw &amp; derived feature groups.</span>

> **Not medical advice.** This is a personal training-readiness & recovery monitoring system, not a diagnosis or injury-prediction tool.


This notebook ingests Garmin data, validates it, and writes the **raw** and **derived** feature groups that the three models consume:

- raw: daily summary, sleep, activity, intra-day epochs
- derived: recovery baselines, training load (EWMA), realtime stress state, recovery episodes

Design rationale for every choice here lives in **`DESIGN_DECISIONS.md`**.

In [ ]:
# Dependencies (hopsworks, scikit-learn, great_expectations, garminconnect) are
# already installed in this environment, so the in-notebook pip install is skipped.
# (Re-enable with: !uv pip install --python $(which python3) 'hopsworks[python,great_expectations]' scikit-learn garminconnect)

## <span style='color:#ff5f27'>📝 Imports &amp; configuration</span>

In [ ]:
import datetime as dt
import warnings

import pandas as pd

from ingestion import get_source
from features import daily_summary, sleep, activity, baselines, training_load, epoch

warnings.filterwarnings("ignore")

# Real-data run: pull this account's own Garmin history via python-garminconnect.
# Credentials are read from Hopsworks secrets in the ingestion cell below.
DEMO_MODE = False
USER_ID = "javier"
# Dynamic window so a scheduled daily run always refreshes through yesterday.
END_DATE = dt.date.today() - dt.timedelta(days=1)   # last complete day
START_DATE = END_DATE - dt.timedelta(days=120)      # ~4 months trailing (stable baselines/EWMA)

## <span style='color:#ff5f27'>📡 1. Ingest raw Garmin data</span>

The ingestion layer is abstracted behind `get_source(...)` so the backend (`python-garminconnect` vs. demo fixtures) is swappable. Both emit the same *canonical raw* schema, so normalization has a single code path.

In [ ]:
import os

# Reuse a cached Garmin OAuth token (avoids repeated logins -> Garmin 429).
os.environ.setdefault("GARMINTOKENS", os.path.expanduser("~/.garminconnect_token"))

if not DEMO_MODE:
    # Garmin credentials come from Hopsworks project secrets, never hard-coded.
    import hopsworks
    hopsworks.login()
    _sec = hopsworks.get_secrets_api()
    os.environ["GARMIN_EMAIL"] = _sec.get_secret("GARMIN_EMAIL").value
    os.environ["GARMIN_PASSWORD"] = _sec.get_secret("GARMIN_PASSWORD").value

source = get_source(demo_mode=DEMO_MODE, seed=42)

raw_daily = source.fetch_daily_summaries(START_DATE, END_DATE)
raw_sleep = source.fetch_sleep(START_DATE, END_DATE)
raw_activities = source.fetch_activities(START_DATE, END_DATE)
raw_epochs = source.fetch_epochs(START_DATE, END_DATE)

print(f"daily={len(raw_daily)} sleep={len(raw_sleep)} "
      f"activities={len(raw_activities)} epochs={len(raw_epochs)}")

In [ ]:
from ingestion import normalize

daily_df = daily_summary.add_derived(normalize.to_daily_summary_df(raw_daily, USER_ID, source.source_name))
sleep_df = sleep.add_derived(normalize.to_sleep_df(raw_sleep, USER_ID, source.source_name))
activity_df = activity.add_load(normalize.to_activity_df(raw_activities, USER_ID, source.source_name))
epoch_df = normalize.to_epoch_df(raw_epochs, USER_ID, source.source_name)

daily_df.head()

## <span style='color:#ff5f27'>🔌 2. Connect to Hopsworks</span>

In [ ]:
import hopsworks

project = hopsworks.login()
fs = project.get_feature_store()

## <span style='color:#ff5f27'>✅ 3. Data validation (Great Expectations)</span>

We attach a **light** expectation suite (~4 rules) to the daily-summary feature group. Hard physiological bounds use a **STRICT** ingestion policy (reject bad batches); soft/observational checks would use **ALWAYS** (ingest-and-flag). See `DESIGN_DECISIONS.md` A6.

In [ ]:
import great_expectations as ge
from great_expectations.core import ExpectationConfiguration

daily_suite = ge.core.ExpectationSuite(expectation_suite_name="garmin_daily_summary_suite")
for col, lo, hi in [("resting_hr", 30, 120), ("avg_stress", 0, 100),
                    ("body_battery_high", 0, 100), ("spo2_avg", 70, 100)]:
    daily_suite.add_expectation(ExpectationConfiguration(
        expectation_type="expect_column_values_to_be_between",
        kwargs={"column": col, "min_value": lo, "max_value": hi},
    ))

## <span style='color:#ff5f27'>🗄️ 4. Raw feature groups</span>

Each raw FG is **online-enabled** with an `event_time` for point-in-time joins. The high-cardinality **epoch** FG uses an online **TTL** window (`ttl` / `online_disk`) so the online store stays bounded while the offline store keeps full history (A5/B5).

In [ ]:
daily_fg = fs.get_or_create_feature_group(
    name="fg_garmin_daily_summary_raw", version=1,
    description="All-day Garmin wellness summary",
    primary_key=["user_id", "summary_date"], event_time="event_time",
    online_enabled=True, expectation_suite=daily_suite,
    statistics_config={"enabled": True, "histograms": True, "correlations": True},
)
daily_fg.insert(daily_df)

sleep_fg = fs.get_or_create_feature_group(
    name="fg_garmin_sleep_raw", version=1, description="Sleep + overnight HRV",
    primary_key=["user_id", "sleep_date"], event_time="event_time", online_enabled=True,
)
sleep_fg.insert(sleep_df)

activity_fg = fs.get_or_create_feature_group(
    name="fg_garmin_activity_raw", version=1, description="Workouts with HR-zone load",
    primary_key=["user_id", "activity_id"], event_time="event_time", online_enabled=True,
)
activity_fg.insert(activity_df)

# Epoch FG: online TTL-bounded window, offline append-only history.
# NOTE: this cluster's NDB engine rejects a TTL column on on-disk storage
# ("TTL column can't be an on-disk column"), so we keep the TTL window in-memory
# (no online_disk) — the offline store still keeps full history.
epoch_fg = fs.get_or_create_feature_group(
    name="fg_garmin_epoch_raw", version=1, description="Intra-day epochs (near-real-time)",
    primary_key=["user_id", "epoch_start_time"], event_time="event_time",
    online_enabled=True, ttl_enabled=True, ttl=dt.timedelta(days=2),
)
epoch_fg.insert(epoch_df)

## <span style='color:#ff5f27'>🧮 5. Derived feature groups</span>

**Baselines** store trailing 7d/28d *means only*, resolved as-of (no same-day leakage). The *delta vs. baseline* is computed later as a model-dependent transform on the feature view (A1). **Training load** uses the EWMA acute/chronic formulation (B3). **Stress state** is keyed by `user_id` so the online store keeps the latest state while the offline store appends full event-time history for the anomaly model (A2/B5).

In [ ]:
# --- Recovery baselines -----------------------------------------------------------
merged = (
    daily_df.rename(columns={"summary_date": "date"})
    .merge(sleep_df.rename(columns={"sleep_date": "date"})[
        ["user_id", "date", "hrv_avg_sleep", "sleep_duration_min", "sleep_debt_min", "sleep_score"]],
        on=["user_id", "date"], how="left")
)
baselines_df = baselines.build_baselines(merged)
baselines_fg = fs.get_or_create_feature_group(
    name="fg_recovery_baselines_daily", version=1, description="Personal recovery baselines",
    primary_key=["user_id", "date"], event_time="event_time", online_enabled=True,
)
baselines_fg.insert(baselines_df)

In [ ]:
# --- Training load (EWMA acute/chronic) -------------------------------------------
daily_load = activity.daily_load(activity_df)
calendar = merged[["user_id", "date"]].drop_duplicates()
daily_load_full = (
    calendar.merge(daily_load, on=["user_id", "date"], how="left")
    .fillna({"load": 0.0, "zone4_5_minutes": 0.0, "n_sessions": 0, "n_strength": 0})
)
load_df = training_load.build_training_load(daily_load_full)
load_fg = fs.get_or_create_feature_group(
    name="fg_training_load_daily", version=1, description="EWMA acute/chronic load + strain",
    primary_key=["user_id", "date"], event_time="event_time", online_enabled=True,
)
load_fg.insert(load_df)

In [ ]:
# --- Realtime stress state (PK=user_id: online=latest, offline=full history) ------
tod_stats = epoch.time_of_day_stats(epoch_df)
stress_state_df = epoch.build_stress_state(epoch_df, tod_stats)

tod_fg = fs.get_or_create_feature_group(
    name="fg_time_of_day_stats", version=1, description="Robust per-hour stress/HR baselines",
    primary_key=["user_id", "hour"], online_enabled=True,
)
tod_fg.insert(tod_stats)

# TTL window kept in-memory (no online_disk) for the same NDB constraint as the epoch FG.
stress_state_fg = fs.get_or_create_feature_group(
    name="fg_stress_state", version=1,
    description="Realtime stress state; online=latest, offline=event-time history",
    primary_key=["user_id"], event_time="event_time",
    online_enabled=True, ttl_enabled=True, ttl=dt.timedelta(days=1),
)
stress_state_fg.insert(stress_state_df)

✅ Raw and derived feature groups are populated. The **recovery-episode** labels and **manual feedback** are built in the next notebook (they need future windows to mature, and feedback for honest evaluation). Continue to **`2_garmin_training_pipeline.ipynb`**.